In [0]:
silver_checkpoint_path = "/Volumes/retail_sales_dev/bronze/checkpoints/sales_silver/checkpoint"

bronze_stream_df = (
    spark.readStream
    .table("retail_sales_dev.bronze.sales_raw")
)

In [0]:
from pyspark.sql.functions import col, to_timestamp, when,try_to_timestamp,lit

silver_df = (
    bronze_stream_df
    # .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))
    .withColumn("InvoiceDate", try_to_timestamp(col("InvoiceDate"), lit("dd-MM-yy H:mm")))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("Price").cast("double"))
    .withColumn("is_cancelled", col("Invoice").startswith("C"))
    .withColumn("has_customer_id", col("CustomerID").isNotNull())
    .withColumn("CustomerID", when(col("CustomerID").isNull(), "UNKNOWN").otherwise(col("CustomerID")))
    .withWatermark("InvoiceDate", "3 days")
    .dropDuplicates(["Invoice", "StockCode", "CustomerID", "Quantity", "InvoiceDate"])
)

In [0]:
query = (
    silver_df.writeStream
    .format("delta")
    .option("checkpointLocation", silver_checkpoint_path)
    .trigger(availableNow=True)
    .toTable("retail_sales_dev.silver.sales_cleaned")
)

query.awaitTermination()

In [0]:
%sql
-- SELECT COUNT(*) FROM retail_sales_dev.bronze.sales_raw; --77628
-- SELECT COUNT(*) FROM retail_sales_dev.silver.sales_cleaned; --76731

-- SELECT has_customer_id, COUNT(*), SUM(Quantity * UnitPrice) AS revenue
-- FROM retail_sales_dev.silver.sales_cleaned
-- GROUP BY has_customer_id;

SELECT is_cancelled, COUNT(*)
FROM retail_sales_dev.silver.sales_cleaned
GROUP BY is_cancelled;

In [0]:
%sql
-- SELECT * FROM retail_sales_dev.silver.sales_cleaned Where InvoiceNo is not null; --76731

SELECT * FROM retail_sales_dev.bronze.sales_raw Where Invoice is not null; --76731

-- SELECT COUNT(*) FROM retail_sales_dev.bronze.sales_raw WHERE _rescued_data IS NOT NULL